In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
video_path = "data/video/YOUR_SUBJECT.mp4"

cap = cv2.VideoCapture(video_path)

In [ ]:
path = "/Users/adityaacharyaresearch/biovid-pain-project/data/video/071709_w_23.mp4"


In [ ]:
cap = cv2.VideoCapture(path)


In [ ]:
import os

print(os.path.exists(path))
print(os.path.getsize(path) if os.path.exists(path) else "FILE NOT FOUND")

In [ ]:
print("Opened:", cap.isOpened())


In [ ]:
print("FPS:", cap.get(cv2.CAP_PROP_FPS))


In [ ]:
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

In [ ]:
ret, frame = cap.read()

In [ ]:
print(ret)

In [ ]:
print(frame.shape)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))

In [ ]:
print(type(frame))

In [ ]:
print(frame.shape)

In [ ]:
print(frame.dtype)

In [ ]:
print(frame[100, 100, 0])  # Blue
print(frame[100, 100, 1])  # Green
print(frame[100, 100, 2])

In [ ]:
import mediapipe as mp

In [ ]:
from mediapipe.tasks.python import vision
from mediapipe.tasks.python import BaseOptions

In [ ]:
from pathlib import Path

model_path = Path("/Users/adityaacharyaresearch/biovid-pain-project/models/face_landmarker.task")

print(model_path.exists())
print(model_path)

In [ ]:
options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(model_path)
    ),
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1
)

In [ ]:
landmarker = vision.FaceLandmarker.create_from_options(options)


In [ ]:
print("Face Landmarker ready!")

In [ ]:
print(model_path)
print(type(model_path))

In [ ]:
options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(model_path)
    ),
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1
)

print(options)

In [ ]:
landmarker = vision.FaceLandmarker.create_from_options(options)

In [ ]:
mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
)

In [ ]:
result = landmarker.detect(mp_image)

In [ ]:
print(len(result.face_landmarks))

In [ ]:
landmarks = result.face_landmarks[0]


In [ ]:
print(type(landmarks))


In [ ]:
print(len(landmarks))


In [ ]:
print(landmarks[10])

In [ ]:
h, w = frame.shape[:2]

In [ ]:
frame.shape

In [ ]:
x_pixel = int(landmarks[10].x*w)
y_pixel = int(landmarks[10].y*h)

In [ ]:
print("Image:", w, "×", h)


In [ ]:
print("Landmark 10:", x_pixel, y_pixel)

In [ ]:
plt.figure(figsize=(12,7))

In [ ]:
plt.figure(figsize=(12,7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.scatter(
    x_pixel,
    y_pixel,
    s=100
)

plt.axis("off");


In [ ]:
plt.scatter(
    x_pixel,
    y_pixel,
    s=100
)

plt.axis("off");

In [ ]:
landmark_ids = [10, 9, 67, 297]

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

for idx in landmark_ids:
    x = int(landmarks[idx].x * w)
    y = int(landmarks[idx].y * h)

    plt.scatter(x, y, s=100)
    plt.text(x + 10, y, str(idx), fontsize=14)

plt.axis("off");

In [ ]:
x0 = int(landmarks[67].x * w)
x1 = int(landmarks[297].x * w)

y0 = int(landmarks[10].y * h)
y1 = int(landmarks[9].y * h)

print("left:", x0)
print("right:", x1)
print("top:", y0)
print("bottom:", y1)

In [ ]:
roi = frame[y0:y1, x0:x1]

In [ ]:
plt.figure(figsize=(12, 7))

plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

plt.plot(
    [x0, x1, x1, x0, x0],
    [y0, y0, y1, y1, y0]
)

plt.axis("off");

In [ ]:
print("Full frame:", frame.shape)
print("ROI:", roi.shape)

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
print(roi[0, 0])

print(roi[0, 0])

print(frame[y0, x0])

print(frame[y0, x0])

print(frame[y0, x0])

In [ ]:
print(frame[y0, x0])

In [ ]:
mean_rgb = roi.mean(axis=(0,1))

In [ ]:
print(mean_rgb)

In [ ]:
print("ROI shape:", roi.shape)
print("Mean B:", mean_rgb[0])
print("Mean G:", mean_rgb[1])
print("Mean R:", mean_rgb[2])

In [ ]:
green = mean_rgb[1]

print(green)

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

In [ ]:
green_signal = []

In [ ]:
green_signal = []

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # detect face
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    )

    result = landmarker.detect(mp_image)

    if len(result.face_landmarks) == 0:
        green_signal.append(np.nan)
        continue

    landmarks = result.face_landmarks[0]

    h, w = frame.shape[:2]

    # forehead boundaries
    x0 = int(landmarks[67].x * w)
    x1 = int(landmarks[297].x * w)
    y0 = int(landmarks[10].y * h)
    y1 = int(landmarks[9].y * h)

    # crop forehead
    roi = frame[y0:y1, x0:x1]

    # average RGB
    mean_rgb = roi.mean(axis=(0, 1))

    # green channel
    green = mean_rgb[1]

    green_signal.append(green)

In [ ]:
green_signal

In [ ]:
elapsed = time.time() - start


In [ ]:
print("Number of measurements:", len(green_signal))

In [ ]:
print("First 10 values:", green_signal[:10])

In [ ]:
green_signal= np.array(green_signal)

In [ ]:
print("First 10 values:", green_signal[:10])


In [ ]:
print(green_signal.shape)

In [ ]:
fps

In [ ]:
time = np.arange(len(green_signal))/25

In [ ]:
time = np.arange(len(green_signal)) / fps

In [ ]:
plt.plot(time, green_signal)

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)

In [ ]:
fps

In [ ]:
green_norm = green_signal/np.mean(green_signal)

In [ ]:
plt.Figure(figsize=(12,5))
plt.plot(time, green_norm)

In [ ]:
plt.plot(time, green_signal)

In [ ]:
plt.plot(time, green_norm)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(time[:250], green_norm[:250])

plt.xlabel("Time (s)")
plt.ylabel("Normalized green")
plt.title("Forehead green signal — first 10 seconds")

plt.show()

In [ ]:
from scipy import signal

green_detrended = signal.detrend(green_norm)

In [ ]:
plt.plot(time, green_signal)

In [ ]:
plt.plot(time, green_detrended)


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(time[:250], green_detrended[:250])

plt.xlabel("Time (s)")
plt.ylabel("Detrended green")
plt.title("Detrended forehead green signal — first 10 seconds")

plt.show()

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt

f, power = signal.periodogram(
    green_detrended,
    fs=fps
)

plt.figure(figsize=(12, 5))

plt.plot(f, power)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.title("Frequency spectrum of forehead green signal")

plt.xlim(0, 4)

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(f, power)

plt.xlim(0.5, 3.0)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.title("Frequency spectrum — heart-rate region")

plt.show()

In [ ]:
window = green_detrended[:int(15 * fps)]

f_window, power_window = signal.periodogram(
    window,
    fs=fps
)

plt.figure(figsize=(12, 5))

plt.plot(f_window, power_window)

plt.xlim(0.5, 3.0)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.title("Frequency spectrum — first 15 seconds")

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(time[:250], red_signal[:250], label="Red")
plt.plot(time[:250], green_signal[:250], label="Green")
plt.plot(time[:250], blue_signal[:250], label="Blue")

plt.xlabel("Time (s)")
plt.ylabel("Mean intensity")
plt.title("RGB signals from forehead")
plt.legend()

plt.show()

In [ ]:
from scipy import signal
import numpy as np

In [ ]:
f, power = signal.periodogram(
    green_detrended,
    fs=fps
)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(f,power)
plt.xlim(0, 4)

In [ ]:
heart_band = (f >= 0.5) & (f <= 3.0)

plt.figure(figsize=(12, 5))

plt.plot(f[heart_band], power[heart_band])

plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.title("Forehead green signal — cardiac frequency range")

plt.show()

In [ ]:
window = green_detrended[:int(15 * fps)]

f_window, power_window = signal.periodogram(
    window,
    fs=fps
)

plt.figure(figsize=(12, 5))

plt.plot(f_window, power_window)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.title("Frequency spectrum — first 15 seconds")

plt.xlim(0.5, 3.0)

plt.show()

In [ ]:
bio = pd.read_csv(
    "/Users/adityaacharyaresearch/biovid-pain-project/data/probe_bio/biosignals_raw/071709_w_23.csv",
    sep="\t"
)

ecg = bio["ecg"].to_numpy()
ecg_time = bio["time"].to_numpy() / 1e6

print(len(ecg))
print(ecg.min(), ecg.max())

bio = pd.read_csv(
    "/Users/adityaacharyaresearch/biovid-pain-project/data/probe_bio/biosignals_raw/071709_w_23.csv",
    sep="\t"
)

ecg = bio["ecg"].to_numpy()
ecg_time = bio["time"].to_numpy() / 1e6

print(len(ecg))
print(ecg.min(), ecg.max())

In [ ]:
import pandas as pd

In [ ]:
bio = pd.read_csv(
    "/Users/adityaacharyaresearch/biovid-pain-project/data/probe_bio/biosignals_raw/071709_w_23.csv",
    sep="\t"
)

ecg = bio["ecg"].to_numpy()
ecg_time = bio["time"].to_numpy() / 1e6

print(len(ecg))
print(ecg.min(), ecg.max())

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(ecg_time[:int(10 * 512)], ecg[:int(10 * 512)])

plt.xlabel("Time (s)")
plt.ylabel("ECG")
plt.title("ECG — first 10 seconds")

plt.show()

In [ ]:
from scipy import signal

peaks, properties = signal.find_peaks(
    ecg,
    distance=int(0.4 * 512),
    height=np.std(ecg)
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(ecg_time[:int(10 * 512)], ecg[:int(10 * 512)])

mask = peaks < int(10 * 512)

plt.scatter(
    ecg_time[peaks[mask]],
    ecg[peaks[mask]],
    s=80
)

plt.xlabel("Time (s)")
plt.ylabel("ECG")
plt.title("Detected R-peaks")

plt.show()

In [ ]:
peak_times = peaks / 512

In [ ]:
rr_intervals = np.diff(peak_times)

print(rr_intervals[:10])

In [ ]:
hr = 60 / rr_intervals

print(hr[:10])
print("Mean HR:", np.mean(hr))

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(time, rgb[:, 0], label="Red")
plt.plot(time, rgb[:, 1], label="Green")
plt.plot(time, rgb[:, 2], label="Blue")

plt.xlabel("Time (s)")
plt.ylabel("Mean pixel value")
plt.title("RGB signals from forehead ROI")
plt.legend()

plt.show()

In [ ]:
rgb_values = []

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # detect face
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    )

    result = landmarker.detect(mp_image)

    if len(result.face_landmarks) == 0:
        rgb_values.append([np.nan, np.nan, np.nan])
        continue

    landmarks = result.face_landmarks[0]

    h, w = frame.shape[:2]

    # forehead boundaries
    x0 = int(landmarks[67].x * w)
    x1 = int(landmarks[297].x * w)
    y0 = int(landmarks[10].y * h)
    y1 = int(landmarks[9].y * h)

    roi = frame[y0:y1, x0:x1]

    mean_rgb = roi.mean(axis=(0, 1))

    rgb_values.append(mean_rgb)

print(rgb.shape)
print(rgb[:5])

In [ ]:
rgb = np.array(rgb_values)

print(rgb.shape)
print(rgb[:5])

In [ ]:
rgb = rgb[:, [2, 1, 0]]

print(rgb[:5])

In [ ]:
window = rgb[:int(10 * fps)]

print(window.shape)